# Clinic Case Study — Phase 1: Single-Nurse Model

**Case study**: Community Health Clinic | **Phase**: 1 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Build and run a single-stage (single-nurse) SimPy model using `simdes`.
2. Verify simulation output against the M/M/1 analytical formula.
3. Interpret utilisation and mean wait as functions of arrival rate.
4. Explain why the model in this phase is an M/M/1 queue in disguise.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
>
> Phase 1 is deliberately simple: one resource, one queue. The goal is to build
> intuition and establish a verified baseline before adding complexity in later phases.

In [ ]:
import sys
from pathlib import Path
# Ensure course/ is on sys.path so the case_studies package is importable
_root = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'simdes').is_dir())
_course = _root / 'course'
if str(_course) not in sys.path:
    sys.path.insert(0, str(_course))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from case_studies.clinic.clinic_model import run_clinic, ClinicParams
from simdes.analysis import confidence_interval

## System Description

In Phase 1 we model the **triage nurse** stage only — the bottleneck in a small
walk-in clinic.  Patients arrive at rate λ = 5 patients/hour throughout an 8-hour
operating day.  A single nurse assesses each patient; assessment time averages 8 minutes.

| Parameter | Value | Unit |
|---|---|---|
| λ (arrival rate) | 5/hr = 1/12 per min | patients/min |
| μ (nurse service rate) | 1/8 | patients/min |
| ρ (utilisation) | λ/μ = (1/12)/(1/8) = 0.667 | — |
| Theoretical Wq | ρ/(μ−λ) = 0.667/(1/8−1/12) = 16 min | minutes |

**Simplification for Phase 1**: we suppress the registration desk
(`n_registration=0` is not supported, so we set registration mean ≈ 0).

In [ ]:
# Phase 1: single nurse, no registration, no exam rooms
p1 = ClinicParams(
    n_registration=1,
    n_nurses=1,
    n_exam_rooms=100,    # effectively unlimited — not the bottleneck
    arrival_rate=5.0 / 60.0,   # 5 patients/hour → per minute
    reg_mean=0.1,              # near-zero registration time
    triage_mean=8.0,
    exam_mean=0.1,             # near-zero exam time
    sim_time=480.0,
)

df = run_clinic(p1, n_reps=30)
df.head()

In [ ]:
# Analytical benchmark
lam = 5.0 / 60.0   # per minute
mu  = 1.0 / 8.0    # per minute
rho = lam / mu
Wq_theory = rho / (mu - lam)   # M/M/1 wait in queue
W_theory  = 1.0 / (mu - lam)   # M/M/1 total time in system

sim_mean, ci_lo, ci_hi = confidence_interval(df['mean_wait_triage'].to_numpy())

print(f'ρ = {rho:.3f}')
print(f'Theoretical Wq = {Wq_theory:.2f} min')
print(f'Simulation Wq:  {sim_mean:.2f} min  95% CI [{ci_lo:.2f}, {ci_hi:.2f}]')
print(f'Theory in CI?  {ci_lo <= Wq_theory <= ci_hi}')

In [ ]:
# Sweep utilisation: vary n_nurses while holding arrival rate fixed
arrival_rates = [3.0, 4.0, 5.0, 6.0, 7.0]   # patients/hour
results = []

for lam_hr in arrival_rates:
    p = ClinicParams(
        n_nurses=1, n_registration=1, n_exam_rooms=100,
        arrival_rate=lam_hr / 60.0, reg_mean=0.1, triage_mean=8.0,
        exam_mean=0.1, sim_time=4800.0,   # longer run for heavy traffic
    )
    df_r = run_clinic(p, n_reps=20)
    mean_wq, lo, hi = confidence_interval(df_r['mean_wait_triage'].to_numpy())
    rho_val = (lam_hr / 60.0) / (1.0 / 8.0)
    results.append({'lam_hr': lam_hr, 'rho': rho_val,
                    'sim_Wq': mean_wq, 'ci_lo': lo, 'ci_hi': hi})

res_df = pd.DataFrame(results)
res_df

In [ ]:
# Hockey-stick plot: Wq vs rho
rho_grid = np.linspace(0.01, 0.96, 200)
Wq_mm1 = (rho_grid / (1.0/8.0)) / (1 - rho_grid)   # M/M/1 formula

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rho_grid, Wq_mm1, 'k--', label='M/M/1 theory', lw=1.5)
ax.errorbar(res_df['rho'], res_df['sim_Wq'],
            yerr=[res_df['sim_Wq']-res_df['ci_lo'], res_df['ci_hi']-res_df['sim_Wq']],
            fmt='o', color='tab:blue', capsize=4, label='Simulation (30 reps)')
ax.set_xlabel(r'Utilisation $\rho$')
ax.set_ylabel('Mean triage wait $W_q$ (min)')
ax.set_title('Phase 1 — Single-nurse clinic: simulation vs. M/M/1')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Summary

The Phase 1 single-nurse model is exactly an M/M/1 queue:
- Poisson arrivals (exponential interarrival times)
- Exponential service times
- One server

Simulation agrees with the analytical result within the 95% confidence interval.
As ρ → 1 (nurse fully loaded), waiting time grows rapidly — the classic hockey-stick shape.

**Key takeaway**: before adding model complexity, verify the simplest case analytically.
This is the first step of every simulation V&V plan.

## Try It Yourself

1. What happens to the CI width as you increase the number of replications from 30 to 100?
2. Set `triage_mean=12.0` (slower nurse). What is the new ρ? Does the simulation still agree with theory?
3. For ρ = 0.9, how long must the simulation run before the transient warm-up period ends?
   Use Welch's method (`simdes.analysis.warmup`) to estimate a warm-up period.